# 수도권제1순환고속도로 — 3D 라벨(spatial × temporal) 예측 데모 노트북

이 노트북은 **(월, 요일타입, 방향)** 단위의 입력 피처 `X`와  
**(구간 × 시간)** 2차원 그리드 라벨 `Y`를 사용하는 **3차원 라벨 회귀 문제**를 시연합니다.

- **입력 X**: `(B, D)` — B는 `(월, 요일타입, 방향)` 그룹 개수, D는 피처 차원(예: 3)
- **라벨 Y**: `(B, S, T)` — S는 구간(seg_idx) 개수, T는 시간대(24시간)
- **모델**: `stnet` 패키지의 `Root` + `ModelConfig/PatchConfig`를 사용하는 spatiotemporal 모델
- **런타임**: Colab(T4) / 로컬 GPU / CPU 모두 지원(장비에 따라 설정 분기)

> ⚠️ 이 노트북은 **문제 정의/차원 정합 시연용**입니다.  
> `stnet` 패키지 설치 상태, 런타임 환경 등에 따라 추가 오류가 발생할 수 있습니다.


## 1. 환경 및 의존성 설정

In [ ]:
# Colab/로컬 공통 의존성 설치 (stnet은 별도 설치 가정)
import sys
import importlib

def _ensure(pkg: str, mod: str | None = None) -> None:
    mod = mod or pkg
    try:
        importlib.import_module(mod)
    except Exception:
        import subprocess
        print(f"[install] {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=True)

# 기본 분석 스택
_ensure("polars")
_ensure("pandas")
_ensure("openpyxl")
_ensure("tensordict")
_ensure("torchrl")
_ensure("fastexcel")
_ensure("xlsxwriter")
_ensure("matplotlib")

import os
import math
from typing import Any, Dict, List, Tuple, Optional

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)
pd.set_option("mode.copy_on_write", True)

# Colab 감지 및 Google Drive 마운트
try:
    import google.colab  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=True)
        print("[info] Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"[warn] Google Drive mount failed: {e!r}")

print(f"IS_COLAB = {IS_COLAB}")

In [ ]:
_ensure("triton")
_ensure("pynvml")
_ensure("transformer-engine-cu12")
_ensure("transformer-engine-torch")

## 2. 입력 엑셀 경로 및 출력 디렉터리 설정

In [ ]:
from typing import Union
import os

# 기본 엑셀 경로 (로컬 실행 기준)
OUT_DIR = os.path.join(os.getcwd(), r"drive/My Drive/")
RAW_PATH = os.path.join(OUT_DIR, r"raw_data.xlsx")
os.chdir(OUT_DIR)

print("EXCEL_PATH:", RAW_PATH)
print("OUT_DIR   :", str(OUT_DIR))

## 3. 원본 엑셀 → Long 포맷 (월, 요일타입, 방향, 구간, 시간, 지표)

In [ ]:
# 원본 엑셀에는 시트별로 '노선', '구간', '방향', '00시'~'23시'가 있음
import re

def parse_sheet_name(name: str) -> Tuple[int, str]:
    """시트명에서 (월, 일종) 추출. 예: '1월 평일' -> (1, '평일')"""
    m = re.search(r"(\d+)월", name)
    if not m:
        raise ValueError(f"시트명에서 월 정보를 찾지 못했습니다: {name}")
    month = int(m.group(1))
    if ("주말" in name) or ("공휴일" in name):
        day_kind = "주말·공휴일"
    else:
        day_kind = "평일"
    return month, day_kind

HOURS = [f"{h:02d}시" for h in range(24)]

# 시트 목록
xlsx = pd.ExcelFile(RAW_PATH)
sheet_names: List[str] = xlsx.sheet_names
print(f"총 시트 수: {len(sheet_names)}")
print("샘플 시트:", sheet_names[:8])

long_parts_pl: List[pl.DataFrame] = []

for sh in sheet_names:
    # 폴라스로 엑셀 로드
    try:
        df_pl = pl.read_excel(str(RAW_PATH), sheet_name=sh)
    except Exception as e:
        print(f"[warn] skip sheet={sh}: {e!r}")
        continue

    # 컬럼 이름 정리
    df_pl = df_pl.rename({c: str(c).strip() for c in df_pl.columns})

    # 필수 컬럼 확인 ("노선" 컬럼도 있는지 체크하면 좋음)
    if not {"구간", "방향"}.issubset(df_pl.columns):
        print(f"[warn] 스킵: 필수 컬럼 없음 (시트={sh})")
        continue

    hour_cols = [c for c in df_pl.columns if c in HOURS]
    if not hour_cols:
        print(f"[warn] 스킵: 시간 컬럼 없음 (시트={sh})")
        continue

    month, day_kind = parse_sheet_name(sh)

    # ✅ [수정 1] "노선" 컬럼을 포함하여 선택
    cols_to_select = ["노선", "구간", "방향"] + hour_cols
    # 만약 엑셀에 '노선' 컬럼이 없다면 에러가 날 수 있으므로, 없는 경우 리터럴로 추가하는 방어 로직 추가 가능
    if "노선" not in df_pl.columns:
        df_pl = df_pl.with_columns(pl.lit("수도권제1순환선").alias("노선"))

    df_pl = df_pl.select(cols_to_select)
    df_pl = df_pl.with_columns(
    [pl.col(c).cast(pl.Float64, strict=False) for c in hour_cols]
)
    df_pl = df_pl.with_columns([
        pl.lit(month).alias("월"),
        pl.lit(day_kind).alias("일종"),
    ])

    # ✅ [수정 2] wide -> long 변환 시 "노선"을 id_vars에 포함
    df_long = df_pl.melt(
        id_vars=["노선", "구간", "방향", "월", "일종"], # <--- 여기 "노선" 추가
        value_vars=hour_cols,
        variable_name="시간",
        value_name="지표",
    )
    long_parts_pl.append(df_long)

assert long_parts_pl, "유효한 시트를 하나도 찾지 못했습니다."

long_pl = pl.concat(long_parts_pl, how="vertical")

# 타입 정리 및 정렬
long_pl = long_pl.with_columns([
    pl.col("시간").cast(pl.Utf8).str.replace("시", "").cast(pl.Int64, strict=False),
    pl.col("지표").cast(pl.Float64, strict=False),
])
long_pl = long_pl.drop_nulls("지표")

print("long_pl shape:", long_pl.shape)
long_df = long_pl
print("long_df shape:", long_df.shape)
display(long_df.head(10))

## 4. ID 인코딩 및 seg_idx 정의 (DIR2ID, DAY2ID, SEG2ID)

In [ ]:
from typing import Dict

# 1. 요일/방향 ID 인코딩
DAY2ID: Dict[str, int] = {"평일": 0, "주말·공휴일": 1}
DIR2ID: Dict[str, int] = {"상행": 0, "하행": 1}

long_df = long_df.with_columns([
    pl.col("일종").replace(DAY2ID).cast(pl.Int64).alias("요일타입_id"),
    pl.col("방향").replace(DIR2ID).cast(pl.Int64).alias("방향_id"),
])

# 2. [핵심 수정] 구간 이름 정규화 (Canonicalization)
# "판교JC-성남IC"(상행)와 "성남IC-판교JC"(하행)를 같은 구간으로 취급하기 위해
# 이름을 가나다순으로 정렬해서 통일합니다.
def normalize_section_name(name):
    # "-"를 기준으로 쪼개서 정렬 후 다시 합침
    parts = sorted(name.split("-"))
    return "-".join(parts)

# 정규화된 이름 컬럼('canonical_section') 생성
long_df = long_df.with_columns(
    pl.col("구간").map_elements(normalize_section_name, return_dtype=pl.Utf8).alias("canonical_section")
)

# 3. 정규화된 이름을 기준으로 ID(seg_idx) 생성
# 이제 상행/하행이 물리적으로 같은 도로라면 같은 seg_idx를 갖습니다.
unique_sections = (
    long_df
    .select("canonical_section")
    .unique(maintain_order=True)  # 원본 등장 순서 유지
    .to_series()
    .to_list()
)

SEG2ID: Dict[str, int] = {sec: idx for idx, sec in enumerate(unique_sections)}
S = len(SEG2ID)

# 4. seg_idx 매핑
long_df = long_df.with_columns(
    pl.col("canonical_section").replace(SEG2ID).cast(pl.Int64).alias("seg_idx")
)

# 5. 데이터 정렬 (물리적 구간 순서대로 정렬됨)
long_df = long_df.sort(["월", "요일타입_id", "방향_id", "seg_idx", "시간"])

print(f"통합된 총 물리 구간 수 S = {S}")
# (예: 기존 148개에서 절반인 74개 정도로 줄어들어야 정상입니다)
long_df.head(10)

## 5. X, Y 데이터 구축

In [ ]:
# long_df: (월, 요일타입_id, 방향_id, seg_idx, 시간, 지표)까지 전처리 끝난 Polars DF라고 가정
# 예: long_df.columns → ["월", "요일타입_id", "방향_id", "seg_idx", "시간", "지표", ...]
S = long_df.select(pl.col("seg_idx")).n_unique()
T = 24  # 시간 0~23
print("S (세그 수) =", S, "T (시간 수) =", T)


# --- 그룹 축 정의: (월, 요일타입_id, 방향_id) ---
group_cols = ["월", "요일타입_id", "방향_id"]

# --- 그룹 인덱스 부여 ---
group_df = (
    long_df
    .select(group_cols)
    .unique()
    .sort(group_cols)
)

X_keys: list[Tuple[int, int, int]] = [
    (int(row[0]), int(row[1]), int(row[2]))
    for row in group_df.to_numpy()
]

B = len(X_keys)
print("B (그룹 수) =", B)
print("예시 키 5개:", X_keys[:5])

In [ ]:
import torch

def build_Y_for_group(df_group: pl.DataFrame, S: int, T: int) -> torch.Tensor:
    """
    df_group: 해당 그룹 (월,요일타입_id,방향_id) 에 해당하는 행만 들어있는 Polars DF
              컬럼: seg_idx, 시간, 지표 포함
    """
    # 0으로 초기화
    import numpy as np
    Y_np = np.zeros((S, T), dtype=np.float32)

    for row in df_group.iter_rows(named=True):
        s = int(row["seg_idx"])
        t = int(row["시간"])
        v = float(row["지표"])
        if 0 <= s < S and 0 <= t < T:
            Y_np[s, t] = v

    return torch.from_numpy(Y_np)  # (S, T)

In [ ]:
train_data: Dict[Tuple[int, int, int], torch.Tensor] = {}

for (month, day_id, dir_id) in X_keys:
    # 해당 그룹만 필터링
    df_group = long_df.filter(
        (pl.col("월") == month) &
        (pl.col("요일타입_id") == day_id) &
        (pl.col("방향_id") == dir_id)
    )

    if df_group.height == 0:
        # 데이터가 없으면 건너뛰거나 0 텐서 유지
        print(f"[warn] 그룹 {(month, day_id, dir_id)} 에 해당하는 데이터가 없습니다.")
        Y_tensor = torch.zeros(S, T, dtype=torch.float32)
    else:
        # Numpy로 변환해서 Y 채움
        Y_np = np.zeros((S, T), dtype=np.float32)
        for row in df_group.iter_rows(named=True):
            s = int(row["seg_idx"])
            t = int(row["시간"])
            v = float(row["지표"])
            if 0 <= s < S and 0 <= t < T:
                Y_np[s, t] = v

        Y_tensor = torch.from_numpy(Y_np)  # (S, T)

    key = (month, day_id, dir_id)
    train_data[key] = Y_tensor

print("len(train_data) =", len(train_data))
some_key = next(iter(train_data.keys()))
print("예시 key:", some_key, "Y.shape:", train_data[some_key].shape)

## 6. STNet 모델 구성 (ModelConfig, PatchConfig, Root)

In [ ]:
from stnet.api.config import PatchConfig, ModelConfig
from stnet.api.io import new_model
from stnet.backend.system import get_device
from stnet.data.transforms import preprocess

device = get_device()
print("Device:", device)

feats, labels, keys, label_shape = preprocess(train_data)
# feats: (B, D)   # D = X 튜플에서 파생된 feature 차원
# labels: (B, S, T)
print("feats.shape  =", feats.shape)
print("labels.shape =", labels.shape)
print("label_shape  =", label_shape)

B2, D = feats.shape
assert label_shape == (S, T)

patch = PatchConfig(
    is_cube=True,
    grid_size_3d=(S, T, 1),
    patch_size_3d=(1, 1, 1),
    use_padding=True,
)

if device.type == "cuda":
    config = ModelConfig(
        device=device,
        patch=patch,
        normalization_method="layernorm",
        depth=1024,
        heads=16,
        mlp_ratio=4.0,
        dropout=0.00,
        drop_path=0.00,
        spatial_depth=4,
        temporal_depth=4,
        spatial_latents=64,
        temporal_latents=64,
        activation_checkpoint=False,
        modeling_type="spatiotemporal",
        compile_mode="reduce-overhead",
    )
else:
    config = ModelConfig(
        device=device,
        patch=patch,
        normalization_method="layernorm",
        depth=256,
        heads=2,
        mlp_ratio=2.0,
        dropout=0.05,
        drop_path=0.05,
        spatial_depth=2,
        temporal_depth=2,
        spatial_latents=32,
        temporal_latents=32,
        activation_checkpoint=True,
        modeling_type="spatiotemporal",
        compile_mode="disabled",
    )

model = new_model(in_dim=D, out_shape=label_shape, config=config).to(device)

## 7. 학습 파라미터 설정 및 모델 훈련

In [ ]:
from stnet.api.run import train

# 샘플 개수 = Dict 엔트리 개수
num_samples = len(train_data)
base_params = {
    "epochs": 100 if device.type == "cuda" else 50,
    "base_lr": 3e-3,
    "weight_decay": 1e-4,
}

print("num_samples:", num_samples)
print("base_params:", base_params)

# ✅ 여기서는 train_data (Dict[X_tuple -> Y_tensor])를 그대로 넘긴다.
model = train(
    model,
    train_data,
    epochs=int(base_params["epochs"]),
    base_lr=float(base_params["base_lr"]),
    weight_decay=float(base_params["weight_decay"]),
    val_frac=0.1,
)

In [ ]:
history = model.history()
print(history)
for n in history:
  print(n)

## 8. 모델 추론

In [ ]:
import torch

from stnet.backend.system import get_device
from stnet.api.run import predict

# 1) train_data로 다시 preprocess 해서 train과 같은 입력 피처/키/레이블 얻기
device = get_device()
print("Device:", device)

infer_samples = {
    x: None
    for x, y in train_data.items()
}

# 2. predict 호출
results = predict(
    model,
    infer_samples,
)

In [ ]:
for x, y in results.items():
  print(x, y)

In [ ]:
import polars as pl
import numpy as np
import xlsxwriter

# 1. ID 역방향 매핑 준비
ID2DAY = {v: k for k, v in DAY2ID.items()}
ID2DIR = {v: k for k, v in DIR2ID.items()}

# 2. [핵심] 저장할 '틀(Template)' 만들기
# 원본 데이터(long_df)에서 메타 정보를 추출하여 정렬
template_df = (
    long_df
    .select(["월", "요일타입_id", "방향_id", "노선", "구간", "방향", "seg_idx"])
    .unique()
    .sort(["월", "요일타입_id", "방향_id", "seg_idx"]) # 원본 순서 정렬
)

# 3. 모델 예측 결과를 딕셔너리로 변환
pred_map = {}
for (month, day_id, dir_id), pred_tensor in results.items():
    if hasattr(pred_tensor, "detach"):
        arr = pred_tensor.detach().cpu().numpy()
    else:
        arr = np.array(pred_tensor)

    n_segs, n_hours = arr.shape
    for s_idx in range(n_segs):
        pred_map[(month, day_id, dir_id, s_idx)] = arr[s_idx, :24]

# 4. 템플릿에 예측값 채워 넣기
final_rows = []

for row in template_df.iter_rows(named=True):
    key = (row["월"], row["요일타입_id"], row["방향_id"], row["seg_idx"])

    # 모델 예측값이 있으면 가져오고, 없으면 0으로 채움
    if key in pred_map:
        pred_vals = pred_map[key]
    else:
        pred_vals = [0.0] * 24

    # 결과 행 생성
    out_item = {
        "노선": row["노선"],
        "구간": row["구간"],
        "방향": row["방향"],
        "월": row["월"],
        "요일_id": row["요일타입_id"]
    }

    # ✅ [수정 포인트] 소수점 반올림(.0000000) 후 정수(Int64) 변환
    for t, val in enumerate(pred_vals):
        # round(val): 반올림 (예: 92.6 -> 93.0, 92.4 -> 92.0)
        # int(...): 정수로 변환 (예: 93.0 -> 93)
        out_item[HOURS[t]] = int(round(val))

    final_rows.append(out_item)

# 5. 최종 데이터프레임 생성
# Polars가 자동으로 정수형(Int64)으로 인식하여 생성합니다.
df_final = pl.DataFrame(final_rows)

# 6. 엑셀 저장
output_filename = r"예측 데이터.xlsx"
workbook = xlsxwriter.Workbook(output_filename)

groups = df_final.select(["월", "요일_id"]).unique().sort(["월", "요일_id"])

for row in groups.iter_rows(named=True):
    m = row["월"]
    d_id = row["요일_id"]
    day_str = ID2DAY[d_id]

    df_sheet = df_final.filter(
        (pl.col("월") == m) & (pl.col("요일_id") == d_id)
    )

    target_cols = ["노선", "구간", "방향"] + HOURS
    df_sheet_save = df_sheet.select(target_cols)

    sheet_name = f"{m}월 {day_str}"

    df_sheet_save.write_excel(
        workbook=workbook,
        worksheet=sheet_name,
        header_format={"bold": True, "align": "center", "border": 1},
        column_totals=False
    )

workbook.close()
print(f"✅ 예측 데이터 저장 완료: '{output_filename}'")